# Clase 6 — NLP I: tokenización y secuenciación de texto

Curso **"Fundamentos de TensorFlow desde cero"** — DIH DATAlife, a partir de la certificación EITC/AI/TFF (European IT Certificate / Artificial Intelligence / TensorFlow Fundamentals).

## Objetivos de aprendizaje de esta clase

- Explicar por qué un modelo necesita convertir texto en números antes de aprender de él
- Tokenizar un corpus de texto y construir su vocabulario
- Convertir frases en secuencias numéricas de longitud homogénea (padding)

Este notebook acompaña al capítulo 08 del manual del curso. Ejecuta cada celda en orden con **Mayús + Intro**.

## Parte 1 — Por qué el texto necesita convertirse en números

Una red neuronal solo sabe multiplicar, sumar y ajustar números: no entiende palabras. La rama de la Inteligencia Artificial (IA) que se ocupa de que las máquinas entiendan y generen lenguaje humano se llama **Procesamiento de Lenguaje Natural (NLP, Natural Language Processing)**. Antes de entrenar cualquier modelo de NLP, el primer paso es siempre convertir el texto en números.

¿Por qué no basta con codificar cada letra (por ejemplo, con ASCII)? Porque "listen" (escuchar) y "silent" (silencioso) tienen exactamente las mismas letras en distinto orden, y sin embargo no significan lo mismo. Codificar **palabras completas**, en cambio, sí conserva el sentido: si "me" es el token 1, "encanta" el token 2, "mi" el token 3 y "perro" el token 4, la frase "me encanta mi perro" se convierte en [1, 2, 3, 4]. Si "gato" es el token 5, "me encanta mi gato" es [1, 2, 3, 5]: comparando ambas secuencias, un modelo ya detecta el parecido entre las dos frases.

## Parte 2 — Tokenizar y construir un vocabulario

**Tokenizar** es partir un texto en unidades (tokens, normalmente palabras) y asignar un número entero a cada una. El conjunto de tokens distintos que conoce el modelo es su **vocabulario**. TensorFlow (TF) ofrece la clase `Tokenizer` para construirlo automáticamente a partir de un corpus de frases.

In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer

sentences = [
    'I love my dog',
    'I love my cat',
    'You love my dog!'
]

# num_words limita el vocabulario a las 100 palabras más frecuentes
tokenizer = Tokenizer(num_words=100)

# fit_on_texts recorre el corpus y construye el vocabulario
tokenizer.fit_on_texts(sentences)

# word_index es el diccionario palabra -> token
word_index = tokenizer.word_index
print(word_index)

Fíjate en dos detalles del vocabulario que acabas de imprimir:

- Todo está en minúsculas, aunque en el texto original había una "I" y una "You" en mayúscula
- "dog" y "dog!" cuentan como la misma palabra: el `Tokenizer` quita la puntuación automáticamente al ajustar el vocabulario

Esto es exactamente el comportamiento que necesitas: no quieres tokens distintos para la misma palabra solo porque cambia la puntuación o las mayúsculas.

## Parte 3 — Palabras que el tokenizador no conoce (OOV)

El vocabulario solo contiene las palabras vistas al llamar a `fit_on_texts`. Si más adelante aparece una palabra nueva, necesitas un plan: si simplemente se ignorase, la frase perdería una posición y el modelo no sabría dónde faltaba información.

La solución es reservar un token especial para las palabras **OOV** (Out Of Vocabulary, fuera de vocabulario). Se lo indicas al `Tokenizer` con el parámetro `oov_token`.

In [ ]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

sentences = [
    'I love my dog',
    'I love my cat',
    'You love my dog!',
    'Do you think my dog is amazing?'
]

# oov_token reserva un token para las palabras que no estén en el vocabulario
tokenizer = Tokenizer(num_words=100, oov_token="<OOV>")
tokenizer.fit_on_texts(sentences)
word_index = tokenizer.word_index

# texts_to_sequences convierte cada frase en su secuencia de tokens
sequences = tokenizer.texts_to_sequences(sentences)

print("Word index:", word_index)
print("Secuencias:", sequences)

Observa que `"<OOV>"` ocupa el token número 1: es la palabra más importante de tener siempre disponible, así que el tokenizador se la reserva antes que a ninguna otra.

## Parte 4 — Igualar longitudes con padding

Las cuatro frases anteriores tienen longitudes distintas (de cuatro a siete palabras), pero una red neuronal espera que todas las entradas de un lote tengan la misma forma (`shape`). La función `pad_sequences` añade ceros donde hace falta para igualar esa longitud.

In [ ]:
# maxlen fija la longitud final de todas las secuencias
# por defecto, el relleno (padding) se añade al principio de la secuencia ('pre')
padded = pad_sequences(sequences, maxlen=5)

print("Secuencias originales:", sequences)
print("Secuencias con padding:")
print(padded)

Vamos a comprobar que el token OOV funciona con frases completamente nuevas, que el tokenizador no ha visto nunca.

In [ ]:
# Frases nuevas, con palabras que no estaban en el corpus de entrenamiento
test_data = [
    'i really love my dog',
    'my dog loves my manatee'
]

test_seq = tokenizer.texts_to_sequences(test_data)
print("Secuencias de prueba:", test_seq)

padded_test = pad_sequences(test_seq, maxlen=10)
print("Con padding:")
print(padded_test)

Palabras como "really", "loves" o "manatee" no estaban en el vocabulario, así que se sustituyen por el token OOV (el número 1) en vez de desaparecer. La frase conserva su longitud y su estructura, aunque se pierda el significado exacto de esas palabras.

Dos parámetros más de `pad_sequences` que conviene conocer:

- `padding='post'` añade los ceros al final de la secuencia en vez de al principio
- `truncating='post'` (el valor por defecto) recorta por el final las frases más largas que `maxlen`; con `truncating='pre'` se recorta por el principio

**Nota sobre cómo ha evolucionado esto:** el material de este curso usa `tensorflow.keras.preprocessing.text.Tokenizer`, la API clásica y la que documenta la certificación EITC/AI/TFF. En versiones recientes de TensorFlow, Google recomienda sustituirla progresivamente por la capa `tf.keras.layers.TextVectorization`, que integra la tokenización y el padding directamente dentro del propio modelo. El concepto de fondo —palabras a números, vocabulario, token OOV, longitud homogénea— es exactamente el mismo; solo cambia la herramienta.

## Resumen de la clase

- Una red neuronal solo procesa números: el NLP empieza siempre por convertir texto en números
- Codificar palabras completas (no letras sueltas) conserva el significado y permite comparar frases
- `Tokenizer` construye el vocabulario con `fit_on_texts` y limpia mayúsculas y puntuación automáticamente
- El token OOV evita que las palabras nuevas descuadren la longitud de la secuencia
- `pad_sequences` iguala la longitud de todas las secuencias añadiendo ceros, con control sobre dónde rellenar (`padding`) y dónde truncar (`truncating`)

En el notebook de práctica vas a tokenizar y generar secuencias con padding a partir de un corpus de frases que defines tú.